# 05 — Batch Processing (Folder to DataFrames)

Processes a folder of mixed front/back card images and splits the results
into two DataFrames — one for front-side fields, one for back-side fields —
since a single image only ever shows one side of the card.

> Note: this cell wasn't executed in the captured session, so no output is
> shown here — the function itself follows the same tested logic as the
> single-card pipeline in notebook 04.

In [ ]:
import os
import pandas as pd

FRONT_ONLY_FIELDS = {'name', 'address', 'birth_date'}
BACK_ONLY_FIELDS = {'gender', 'religion', 'martial_state', 'job', 'education', 'husband', 'issue_date', 'expire_date'}

FRONT_COLUMNS = ['source_image', 'name', 'national_id', 'address', 'birth_date', 'country', 'doc_type']
BACK_COLUMNS = ['source_image', 'national_id', 'gender', 'religion', 'martial_state', 'job', 'education', 'husband', 'issue_date', 'expire_date']


def process_single_image(image_path, model_crop, model_fields, ocr_model):
    """Runs the full pipeline (crop -> detect fields -> preprocess -> OCR) on one image."""
    cropped_img, card_type = crop_card(image_path, model_crop)
    if cropped_img is None:
        return None, None

    fields = detect_fields(cropped_img, model_fields)
    if len(fields) == 0:
        return None, None

    row = {"source_image": os.path.basename(image_path)}

    for field_name, data in fields.items():
        if field_name in CONSTANT_FIELDS:
            row[field_name] = CONSTANT_FIELDS[field_name]
            continue
        if field_name == "image":
            row[field_name] = "(photo)"
            continue

        processed_img = preprocess_field(data['image'], field_name)
        texts = extract_text_ordered(processed_img, ocr_model)
        row[field_name] = " ".join(texts) if texts else "not read"

    return row, set(fields.keys())


def build_dataframes_from_folder(folder_path, model_crop, model_fields, ocr_model):
    """Processes every image in a folder, splitting results into front/back DataFrames."""
    front_rows, back_rows, skipped_files = [], [], []

    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Found {len(image_files)} images")

    for filename in image_files:
        image_path = os.path.join(folder_path, filename)
        row, detected_fields = process_single_image(image_path, model_crop, model_fields, ocr_model)

        if row is None:
            skipped_files.append(filename)
            print(f"{filename}: skipped (no card/fields detected)")
            continue

        is_front = len(detected_fields & FRONT_ONLY_FIELDS) > 0
        is_back = len(detected_fields & BACK_ONLY_FIELDS) > 0

        if is_front:
            front_rows.append({col: row.get(col, "-") for col in FRONT_COLUMNS})
            print(f"{filename}: front")
        elif is_back:
            back_rows.append({col: row.get(col, "-") for col in BACK_COLUMNS})
            print(f"{filename}: back")
        else:
            skipped_files.append(filename)
            print(f"{filename}: could not classify")

    df_front = pd.DataFrame(front_rows)
    df_back = pd.DataFrame(back_rows)

    print(f"\nSummary: front={len(df_front)}, back={len(df_back)}, skipped={len(skipped_files)}")
    return df_front, df_back

### Run on a folder of test images

In [ ]:
folder_path = "/kaggle/input/datasets/stud20230837/test-imge/test_images"

df_front, df_back = build_dataframes_from_folder(folder_path, model_crop, model_fields, ocr)

print("=== Front face ===")
display(df_front)

print("=== Back face ===")
display(df_back)